In [0]:
df_shipments_bronze = (
    spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv("/Volumes/workspace/default/raw_data/shipments_raw.csv")
)

display(df_shipments_bronze)

In [0]:
df_shipments_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.pharma_bronze.bronze_shipments")

In [0]:
from pyspark.sql.functions import col

df_shipments_bronze.groupBy("shipment_id") \
    .count() \
    .filter(col("count") > 1) \
    .show()

In [0]:
po_reference = spark.table(
    "workspace.pharma_bronze.bronze_purchase_orders"
).select(
    "po_id",
    col("drug_id").alias("po_drug_id"),
    col("supplier_id").alias("po_supplier_id"),
    col("order_date").alias("po_order_date"),
    col("quantity_ordered").alias("po_quantity_ordered")
)

In [0]:
df_shipments_silver = df_shipments_bronze.join(
    po_reference,
    on="po_id",
    how="inner"
)

In [0]:
df_shipments_silver = df_shipments_silver.withColumn(
    "drug_id",
    col("po_drug_id")
)

In [0]:
df_shipments_silver = df_shipments_silver.withColumn(
    "supplier_id",
    col("po_supplier_id")
)

In [0]:
from pyspark.sql.functions import when

In [0]:
df_shipments_silver = df_shipments_silver.withColumn(
    "shipment_date",
    when(
        col("shipment_date") < col("po_order_date"),
        col("po_order_date")
    ).otherwise(col("shipment_date"))
)

In [0]:
df_shipments_silver = df_shipments_silver.withColumn(
    "quantity_shipped",
    when(
        col("quantity_shipped") > col("po_quantity_ordered"),
        col("po_quantity_ordered")
    ).otherwise(col("quantity_shipped"))
)

In [0]:
df_shipments_silver = df_shipments_silver.drop(
    "po_drug_id",
    "po_supplier_id",
    "po_order_date",
    "po_quantity_ordered"
)

In [0]:
from pyspark.sql.functions import current_timestamp

df_shipments_silver = df_shipments_silver.withColumn(
    "ingestion_timestamp",
    current_timestamp()
)

In [0]:
display(df_shipments_silver)

In [0]:
from delta.tables import DeltaTable

delta_shipments_target = DeltaTable.forName(
    spark,
    "workspace.pharma_silver.silver_shipments"
)

(
    delta_shipments_target.alias("target")
    .merge(
        df_shipments_silver.alias("source"),
        "target.shipment_id = source.shipment_id"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

In [0]:
df_shipments_final = spark.table(
    "workspace.pharma_silver.silver_shipments"
)

print("Shipments Silver records:", df_shipments_final.count())

print(
    "Duplicate shipment IDs:",
    df_shipments_final
    .groupBy("shipment_id")
    .count()
    .filter(col("count") > 1)
    .count()
)

In [0]:
df_shipments_silver.groupBy("shipment_id") \
    .count() \
    .filter(col("count") > 1) \
    .show()

In [0]:
print("Total shipment records:", df_shipments_silver.count())

print(
    "Duplicate shipment IDs:",
    df_shipments_silver
    .groupBy("shipment_id")
    .count()
    .filter(col("count") > 1)
    .count()
)

print(
    "Null shipment IDs:",
    df_shipments_silver
    .filter(col("shipment_id").isNull())
    .count()
)


In [0]:
null_shipment_ids = df_shipments_silver.filter(
    col("shipment_id").isNull()
).count()

print("NULL Shipment IDs:", null_shipment_ids)

In [0]:
duplicate_shipment_ids = (
    df_shipments_silver
    .groupBy("shipment_id")
    .count()
    .filter(col("count") > 1)
    .count()
)

print("Duplicate Shipment IDs:", duplicate_shipment_ids)

In [0]:
negative_shipped_quantity = df_shipments_silver.filter(
    col("quantity_shipped") < 0
).count()

print("Negative Shipped Quantity:", negative_shipped_quantity)